In [ ]:
import mne
import numpy as np
import matplotlib.pyplot as plt
import glob
import os

Labeling

In [ ]:
DATA_DIR_A = "SetA"
DATA_DIR_B = "SetB"
DATA_DIR_C = "SetC"
DATA_DIR_D = "SetD"
DATA_DIR_E = "SetE"

LABEL_C1 = 0 #A
LABEL_C2 = 0 #B
LABEL_C3 = 0 #C
LABEL_C4 = 0 #D
LABEL_C5 = 1 #E

Loading the data

In [28]:


def load_dataset(folder_path, label):
    data = []
    labels = []

    for file in os.listdir(folder_path):
        file_path = os.path.join(folder_path, file)

        # each file is 1 signal that is 4097 samples in Bonn dataset (23.6*Fs)
        signal = np.loadtxt(file_path)

        data.append(signal)
        labels.append(label)

    return np.array(data), np.array(labels)



X_A, Y_A = load_dataset(DATA_DIR_A, LABEL_C1)
X_B, Y_B = load_dataset(DATA_DIR_B, LABEL_C2)
X_C, Y_C = load_dataset(DATA_DIR_C, LABEL_C3)
X_D, Y_D = load_dataset(DATA_DIR_D, LABEL_C4)
X_E, Y_E = load_dataset(DATA_DIR_E, LABEL_C5)


X = np.concatenate([X_A, X_B, X_C, X_D, X_E], axis=0)
Y= np.concatenate([Y_A, Y_B, Y_C, Y_D, Y_E], axis=0)


Feature extraction     ( Features : mean+ std+ energy+ entropy )

In [29]:
import pywt

# Shannon entropy
def compute_entropy(signal):
    hist, _ = np.histogram(signal, bins=100, density=True)
    hist = hist + 1e-6  # avoid log(0)
    return -np.sum(hist * np.log2(hist))


def extract_dwt_features(signal, wavelet='db6', level=5):
    coeffs = pywt.wavedec(signal, wavelet, level=level)

    features = []


    for c in coeffs:
        features.append(np.mean(c))
        features.append(np.std(c))
        features.append(np.sum(c**2))  # energy
        features.append(compute_entropy(c))

    return np.array(features)

In [30]:
# Case A-E
X_AE = np.concatenate([X_A, X_E], axis=0)
Y_AE = np.concatenate([
    np.zeros(len(X_A)),   # Class 0 → A (non-seizure)
    np.ones(len(X_E))     # Class 1 → E (seizure)
])

X_AE_features = np.array([extract_dwt_features(x) for x in X_AE])

# X_AE.shape → (samples, features)
# Y_AE.shape → (samples,)

In [31]:
# Case B-E
X_BE = np.concatenate([X_B, X_E], axis=0)
Y_BE = np.concatenate([
    np.zeros(len(X_B)),   # Class 0 → B (non-seizure)
    np.ones(len(X_E))     # Class 1 → E (seizure)
])

X_BE_features = np.array([extract_dwt_features(x) for x in X_BE])

# X_BE.shape → (samples, features)
# Y_BE.shape → (samples,)

In [32]:
# Case C-E
X_CE = np.concatenate([X_C, X_E], axis=0)
Y_CE = np.concatenate([
    np.zeros(len(X_C)),   # Class 0 → C (non-seizure)
    np.ones(len(X_E))     # Class 1 → E (seizure)
])

X_CE_features = np.array([extract_dwt_features(x) for x in X_CE])

# X_CE.shape → (samples, features)
# Y_CE.shape → (samples,)

In [33]:
# Case D-E
X_DE = np.concatenate([X_D, X_E], axis=0)
Y_DE = np.concatenate([
    np.zeros(len(X_D)),   # Class 0 → D (non-seizure)
    np.ones(len(X_E))     # Class 1 → E (seizure)
])

X_DE_features = np.array([extract_dwt_features(x) for x in X_DE])

# X_DE.shape → (samples, features)
# Y_DE.shape → (samples,)

In [34]:
# Case ABCD-E
X_ABCD = np.concatenate([X_A, X_B, X_C, X_D], axis=0)

X_ABCDE = np.concatenate([X_A, X_B, X_C, X_D, X_E], axis=0)
Y_ABCDE = np.concatenate([
    np.zeros(len(X_ABCD)),   # Class 0 → ABCD (non-seizure)
    np.ones(len(X_E))                           # Class 1 → E (seizure)
])

X_ABCDE_features = np.array([extract_dwt_features(x) for x in X_ABCDE])

# X_ABCDE.shape → (samples, features)
# Y_ABCDE.shape → (samples,)

Train/Test Split on features

In [35]:
from sklearn.model_selection import train_test_split

X_AE_train, X_AE_test, Y_AE_train, Y_AE_test = train_test_split(
    X_AE_features, Y_AE,
    test_size=0.2,
    random_state=42,
    stratify=Y_AE
)

X_BE_train, X_BE_test, Y_BE_train, Y_BE_test = train_test_split(
    X_BE_features, Y_BE,
    test_size=0.2,
    random_state=42,
    stratify=Y_BE
)

X_CE_train, X_CE_test, Y_CE_train, Y_CE_test = train_test_split(
    X_CE_features, Y_CE,
    test_size=0.2,
    random_state=42,
    stratify=Y_CE
)

X_DE_train, X_DE_test, Y_DE_train, Y_DE_test = train_test_split(
    X_DE_features, Y_DE,
    test_size=0.2,
    random_state=42,
    stratify=Y_DE
)

X_ABCDE_train, X_ABCDE_test, Y_ABCDE_train, Y_ABCDE_test = train_test_split(
    X_ABCDE_features, Y_ABCDE,
    test_size=0.2,
    random_state=42,
    stratify=Y_ABCDE
)

Classification (Naive Bayes classifier)

In [36]:
from sklearn.naive_bayes import GaussianNB

model_AE= GaussianNB()
model_AE.fit(X_AE_train, Y_AE_train)

model_BE= GaussianNB()
model_BE.fit(X_BE_train, Y_BE_train)

model_CE= GaussianNB()
model_CE.fit(X_CE_train, Y_CE_train)

model_DE= GaussianNB()
model_DE.fit(X_DE_train, Y_DE_train)

model_ABCDE= GaussianNB()
model_ABCDE.fit(X_ABCDE_train, Y_ABCDE_train)

,"priors priors: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None
,"var_smoothing var_smoothing: float, default=1e-9Portion of the largest variance of all features that is added tovariances for calculation stability... versionadded:: 0.20",1e-09


Prediction/ Evaluation

In [37]:
#prediction
Y_AE_pred = model_AE.predict(X_AE_test)
Y_BE_pred = model_BE.predict(X_BE_test)
Y_CE_pred = model_CE.predict(X_CE_test)
Y_DE_pred = model_DE.predict(X_DE_test)
Y_ABCDE_pred = model_ABCDE.predict(X_ABCDE_test)

In [38]:
#Evaluation
from sklearn.metrics import confusion_matrix

tn, fp, fn, tp = confusion_matrix(Y_AE_test, Y_AE_pred).ravel()
accuracy_AE = (tp + tn) / (tp + tn + fp + fn)

tn, fp, fn, tp = confusion_matrix(Y_BE_test, Y_BE_pred).ravel()
accuracy_BE = (tp + tn) / (tp + tn + fp + fn)

tn, fp, fn, tp = confusion_matrix(Y_CE_test, Y_CE_pred).ravel()
accuracy_CE = (tp + tn) / (tp + tn + fp + fn)

tn, fp, fn, tp = confusion_matrix(Y_DE_test, Y_DE_pred).ravel()
accuracy_DE = (tp + tn) / (tp + tn + fp + fn)

tn, fp, fn, tp = confusion_matrix(Y_ABCDE_test, Y_ABCDE_pred).ravel()
accuracy_ABCDE = (tp + tn) / (tp + tn + fp + fn)

print("Accuracy(A-E):", accuracy_AE* 100)
print("Accuracy(B-E):", accuracy_BE* 100)
print("Accuracy(C-E):", accuracy_CE* 100)
print("Accuracy(D-E):", accuracy_DE* 100)
print("Accuracy(ABCD-E):", accuracy_ABCDE* 100)



Accuracy(A-E): 100.0
Accuracy(B-E): 97.5
Accuracy(C-E): 100.0
Accuracy(D-E): 97.5
Accuracy(ABCD-E): 95.0
